# Fase 2: Engenharia de Features e Pré-processamento
**Projeto:** Sistema NDR com ML em Duas Camadas (TCC)
**Dataset:** CICIDS 2017 (Versão Limpa)

## Objetivos deste Notebook:
1. Carregar o dataset limpo gerado na Fase 1.
2. **Remoção de Colinearidade**: Excluir features com correlação de Pearson extrema (> 0.95).
3. **Data Splitting**: Separar os dados em Treino e Teste de forma estratificada.
4. **Padronização**: Aplicar o `StandardScaler` de forma correta (apenas no treino) para evitar Data Leakage.
5. Salvar os artefatos preparados para o treinamento dos modelos.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os
import gc
import joblib

# Conectando com o Google Drive
from google.colab import drive
drive.mount('/content/drive')

caminho_arquivo = '/content/drive/MyDrive/TCC2/cicids2017_limpo.parquet'

print("Carregando o dataset limpo. O Parquet é bem rápido...")
df = pd.read_parquet(caminho_arquivo)

print(f"Dataset carregado com sucesso!")
print(f"Dimensões: {df.shape}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Carregando o dataset limpo. O Parquet é bem rápido...
Dataset carregado com sucesso!
Dimensões: (2520751, 72)


### 2.1 Remoção de Features com Alta Correlação (> 0.95)
Features altamente correlacionadas (ex: Total Fwd Packets e Subflow Fwd Packets) fornecem exatamente a mesma informação para a rede neural ou árvore de decisão.

Manter ambas só aumenta o tempo de treinamento e o tempo de inferência (crítico para um NDR em tempo real). Vamos varrer a matriz de correlação e remover uma feature de cada par que ultrapasse 95% de semelhança.


In [2]:
# Pegando apenas as colunas numéricas (ignorando a label original)
cols_numericas = df.columns.difference(['Label'])

# Calculando a matriz de correlação absoluta (leva de 1 a 2 minutos)
print("Calculando matriz de correlação cruzada...")
matriz_corr = df[cols_numericas].corr().abs()

# Selecionando o triângulo superior da matriz para não testar o mesmo par duas vezes
upper_tri = matriz_corr.where(np.triu(np.ones(matriz_corr.shape), k=1).astype(bool))

# Encontrando colunas onde a correlação é maior que 0.95
to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > 0.95)]

print(f"\nForam encontradas {len(to_drop)} features altamente redundantes (>95% de correlação).")
print("Features a serem removidas:")
print(to_drop)

# Removendo do DataFrame principal
df.drop(columns=to_drop, inplace=True)
print(f"\nFeatures redundantes removidas. Novo shape: {df.shape}")


Calculando matriz de correlação cruzada...

Foram encontradas 23 features altamente redundantes (>95% de correlação).
Features a serem removidas:
['Bwd Packet Length Max', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Fwd Header Length.1', 'Fwd IAT Max', 'Fwd IAT Total', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Fwd Packets/s', 'Fwd URG Flags', 'Idle Max', 'Idle Mean', 'Idle Min', 'Packet Length Mean', 'Packet Length Std', 'RST Flag Count', 'SYN Flag Count', 'Subflow Bwd Packets', 'Subflow Fwd Packets', 'Total Backward Packets', 'Total Fwd Packets', 'Total Length of Bwd Packets', 'Total Length of Fwd Packets']

Features redundantes removidas. Novo shape: (2520751, 49)


### 2.2 Preparação dos Targets e Split dos Dados (Semente 42)
Aqui nós vamos:
1. Isolar os features (`X`) das respostas (`y`).
2. Fazer o split de 80% para treino e 20% para teste da **Camada 1**.
3. Separar apenas os ataques, preparar o `LabelEncoder` da **Camada 2**, e fazer o split de 80/20.
Em ambos os casos usaremos `stratify` para garantir que a proporção de ataques se mantenha igual no treino e no teste.

In [3]:
from sklearn.preprocessing import LabelEncoder

SEED_1 = 42
TEST_SIZE = 0.2

print("Preparando variáveis globais...")
X = df.drop(columns=['Label', 'is_attack', 'attack_type_encoded'], errors='ignore')

# ==========================================
# CAMADA 1: Binária (Normal vs Ataque)
# ==========================================
y_c1 = df['is_attack']

print(f"\n--- Fazendo o Split da Camada 1 (Binária) ---")
X_train_c1, X_test_c1, y_train_c1, y_test_c1 = train_test_split(
    X, y_c1, test_size=TEST_SIZE, random_state=SEED_1, stratify=y_c1
)
print(f"Treino Camada 1: {X_train_c1.shape[0]} amostras")
print(f"Teste Camada 1:  {X_test_c1.shape[0]} amostras")


# ==========================================
# CAMADA 2: Multiclasse (Somente Ataques)
# ==========================================
df_attacks = df[df['is_attack'] == 1].copy()
X_attacks = df_attacks.drop(columns=['Label', 'is_attack', 'attack_type_encoded'], errors='ignore')
y_c2_raw = df_attacks['Label']

# Fazendo o Label Encode oficial aqui (e guardando ele para exportar depois!)
le_attacks = LabelEncoder()
y_c2 = le_attacks.fit_transform(y_c2_raw)

print(f"\n--- Fazendo o Split da Camada 2 (Multiclasse) ---")
X_train_c2, X_test_c2, y_train_c2, y_test_c2 = train_test_split(
    X_attacks, y_c2, test_size=TEST_SIZE, random_state=SEED_1, stratify=y_c2
)
print(f"Treino Camada 2: {X_train_c2.shape[0]} amostras")
print(f"Teste Camada 2:  {X_test_c2.shape[0]} amostras")

# Limpando memória
del df
del df_attacks
gc.collect()


Preparando variáveis globais...

--- Fazendo o Split da Camada 1 (Binária) ---
Treino Camada 1: 2016600 amostras
Teste Camada 1:  504151 amostras

--- Fazendo o Split da Camada 2 (Multiclasse) ---
Treino Camada 2: 340555 amostras
Teste Camada 2:  85139 amostras


7

### 2.3 Normalização (StandardScaler) sem Data Leakage
Algoritmos baseados em distância (como KNN e SVM) e redes neurais (MLP) são muito sensíveis à escala dos dados (ex: 'Total Fwd Packets' vai de 0 a milhares, enquanto 'Fwd PSH Flags' vai de 0 a 1).
Vamos usar o `StandardScaler` para deixar as features com média 0 e variância 1.

**Nota Crucial:** O `fit` (ajuste) deve acontecer **apenas** nos dados de treino. Os dados de teste recebem apenas o `transform`.


In [4]:
# Instanciando dois scalers diferentes (um para cada camada)
scaler_c1 = StandardScaler()
scaler_c2 = StandardScaler()

print("Padronizando os dados da Camada 1...")
# FIT_TRANSFORM no Treino, apenas TRANSFORM no Teste
X_train_c1_scaled = scaler_c1.fit_transform(X_train_c1)
X_test_c1_scaled  = scaler_c1.transform(X_test_c1)

print("Padronizando os dados da Camada 2...")
# FIT_TRANSFORM no Treino, apenas TRANSFORM no Teste
X_train_c2_scaled = scaler_c2.fit_transform(X_train_c2)
X_test_c2_scaled  = scaler_c2.transform(X_test_c2)

print("\nPadronização finalizada! Ausência de Data Leakage garantida.")


Padronizando os dados da Camada 1...
Padronizando os dados da Camada 2...

Padronização finalizada! Ausência de Data Leakage garantida.


### 2.5 Exportação dos Artefatos para Treinamento
Para não rodarmos todo o pipeline de pré-processamento novamente, vamos salvar todos os arrays gerados (Treino e Teste de ambas as camadas), bem como os "Moldes" de transformação (`scaler_c1`, `scaler_c2` e o `le_attacks`).
O `joblib` é a biblioteca padrão e super eficiente para isso.


In [7]:
# Criando uma sub-pasta no seu drive para ficar organizado
pasta_artefatos = '/content/drive/MyDrive/TCC2/artefatos_treino'
os.makedirs(pasta_artefatos, exist_ok=True)

print("Exportando artefatos da Camada 1...")
joblib.dump(X_train_c1_scaled, f'{pasta_artefatos}/X_train_c1.joblib')
joblib.dump(X_test_c1_scaled,  f'{pasta_artefatos}/X_test_c1.joblib')
joblib.dump(y_train_c1,        f'{pasta_artefatos}/y_train_c1.joblib')
joblib.dump(y_test_c1,         f'{pasta_artefatos}/y_test_c1.joblib')
joblib.dump(scaler_c1,         f'{pasta_artefatos}/scaler_c1.joblib')

print("Exportando artefatos da Camada 2...")
joblib.dump(X_train_c2_scaled, f'{pasta_artefatos}/X_train_c2.joblib')
joblib.dump(X_test_c2_scaled,  f'{pasta_artefatos}/X_test_c2.joblib')
joblib.dump(y_train_c2,        f'{pasta_artefatos}/y_train_c2.joblib')
joblib.dump(y_test_c2,         f'{pasta_artefatos}/y_test_c2.joblib')
joblib.dump(scaler_c2,         f'{pasta_artefatos}/scaler_c2.joblib')

print("Exportando extras...")
joblib.dump(le_attacks, f'{pasta_artefatos}/label_encoder_attacks.joblib')
joblib.dump(list(X.columns), f'{pasta_artefatos}/feature_names.joblib') # Vital para deploy!

print("\nTodos os artefatos foram salvos com sucesso em:")
print(pasta_artefatos)

Exportando artefatos da Camada 1...
Exportando artefatos da Camada 2...
Exportando extras...

Todos os artefatos foram salvos com sucesso em:
/content/drive/MyDrive/TCC2/artefatos_treino
